# Clase 10 — Word embeddings

**Curso:** NLP Clásico con Python · **Bloque 4:** Significado y búsqueda
**Tiempo estimado:** 110 minutos · **Capítulo del manual:** `manual/10_word_embeddings.md`

## Qué vas a ver en esta clase

- La **hipótesis distribucional** ("una palabra se conoce por sus vecinas") y el paso de vectores **dispersos** (recuentos de contexto) a vectores **densos**, con un ejemplo que se calcula a mano.
- La **similitud del coseno** entre dos vectores, calculada a mano y con NumPy.
- **Word2Vec** (skip-gram y CBOW, con la ventana como parámetro) y **fastText** con `gensim`, sobre 3-4 obras de dominio público por idioma.
- **Vecinos más próximos** y **analogías** como demostración de los **límites**, no como resultado: con unos cientos de miles de palabras los vectores son **ruidosos**, y lo medimos con otras semillas y otros parámetros.
- **fastText** ante palabras raras, **palabras fuera de vocabulario** y **erratas**.
- Límites: **un vector por palabra**, sesgos del corpus, poco texto, y una comparación con un **corpus sintético moderno**.

**Requisitos:** clases 4 (bolsa de palabras, vectores de recuentos) y 9 (temas: otro modo de resumir el significado); la 11 seguirá con la búsqueda clásica. **Librerías:** `gensim`, `numpy`, `pandas`, `matplotlib`.
**Datos:** `datos/gutenberg/embeddings_es_obras.txt` y `datos/gutenberg/embeddings_it_obras.txt` (Quijote, La Regenta, Marianela y Lazarillo; Promessi sposi, Pinocchio, La vita militare y Novelle; procedencia en `datos/LEEME.md`), `datos/resenas_*.csv` y `datos/tickets_*.csv` (en Colab, sube la carpeta `datos/`). Los textos literarios son de dominio público y de hace uno a cuatro siglos: **no representan el español ni el italiano de hoy**. Todas las cifras son un *ejemplo ilustrativo*. **Duración de la ejecución:** unos dos minutos.

---
## Celda 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("gensim", "gensim")
print("Librerías listas.")

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec, FastText

warnings.filterwarnings("ignore")
random.seed(42)                          # semilla fija
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def frases_de_texto(texto):
    """Minúsculas, corte en frases y solo letras; descarta las frases de menos de 3 palabras."""
    salida = []
    for trozo in re.split(r"[.!?;:]+\s|\n+", texto.lower()):
        palabras = re.findall(r"[^\W\d_]+", trozo)
        if len(palabras) >= 3:
            salida.append(palabras)
    return salida

def entrenar_w2v(frases, sg=1, window=5, seed=1, epochs=5, vector_size=50, min_count=5):
    """Word2Vec reproducible: workers=1 y semilla fija (sg=1 skip-gram, sg=0 CBOW)."""
    return Word2Vec(frases, vector_size=vector_size, window=window, min_count=min_count, sg=sg,
                    epochs=epochs, workers=1, seed=seed)

def entrenar_ft(frases, sg=1, window=5, seed=1, epochs=5, vector_size=50, min_count=5):
    """fastText reproducible, con n-gramas de caracteres de 3 a 5."""
    return FastText(frases, vector_size=vector_size, window=window, min_count=min_count, sg=sg,
                    epochs=epochs, workers=1, seed=seed, min_n=3, max_n=5)

def coseno(a, b):
    """Similitud del coseno entre dos vectores."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def vecinos(modelo, palabra, n=6):
    """Las n palabras más cercanas por coseno."""
    return [w for w, _ in modelo.wv.most_similar(palabra, topn=n)]

def frecuentes(frases, modelo, n=200, largo=4):
    """Las n palabras más frecuentes (de al menos `largo` letras) que están en el vocabulario del modelo."""
    cuenta = Counter(w for f in frases for w in f)
    return [w for w, _ in cuenta.most_common() if len(w) >= largo and w in modelo.wv.key_to_index][:n]

def solape(m1, m2, palabras, k=10):
    """Fracción media de vecinos (top k) que comparten dos modelos: 1 = idénticos, 0 = ninguno."""
    valores = []
    for w in palabras:
        a, b = set(vecinos(m1, w, k)), set(vecinos(m2, w, k))
        valores.append(len(a & b) / k)
    return float(np.mean(valores))

def rango_analogia(modelo, a, b, c, objetivo):
    """a - b + c: (posición del objetivo entre los candidatos, primer candidato). Excluye a, b y c."""
    if objetivo not in modelo.wv.key_to_index:
        return None, None
    sims = modelo.wv.most_similar(positive=[a, c], negative=[b], topn=None)
    excluir = {modelo.wv.key_to_index[w] for w in (a, b, c)}
    candidatos = [i for i in np.argsort(-sims) if i not in excluir]
    return candidatos.index(modelo.wv.key_to_index[objetivo]) + 1, modelo.wv.index_to_key[candidatos[0]]

def frases_modernas(idioma):
    """Frases del corpus sintético moderno (reseñas + tickets, columna texto_limpio)."""
    salida = []
    for nombre in (f"resenas_{idioma}.csv", f"tickets_{idioma}.csv"):
        for texto in pd.read_csv(cargar_datos(nombre))["texto_limpio"]:
            for trozo in re.split(r"[.!?;:]+\s", str(texto).lower()):
                palabras = re.findall(r"[^\W\d_]+", trozo)
                if len(palabras) >= 3:
                    salida.append(palabras)
    return salida

print("Funciones auxiliares cargadas.")

---
## 1. Similitud del coseno, a mano

Un embedding es un **vector** por palabra. Para decir que dos palabras "se parecen" comparamos sus vectores con el **coseno** del ángulo que forman:

$$\cos(a,b)=\frac{a\cdot b}{\|a\|\,\|b\|}$$

Va de −1 (sentidos opuestos) a 1 (misma dirección); 0 significa ortogonales. Ejemplo pequeño (*ejemplo ilustrativo*): `a = (3, 4)` y `b = (4, 3)`. Antes de ejecutar: `a·b = 12 + 12 = 24`, `‖a‖ = ‖b‖ = 5`, luego `cos = 24/25 = 0,96`.

In [ ]:
a, b = np.array([3, 4]), np.array([4, 3])
print("a·b =", int(a @ b), "| ‖a‖ =", np.linalg.norm(a), "| ‖b‖ =", np.linalg.norm(b))
print("cos(a, b) =", coseno(a, b))
print("cos(a, (-4, 3)) =", coseno(a, [-4, 3]), "| cos(a, -a) =", coseno(a, -a))
assert np.isclose(coseno(a, b), 0.96) and np.isclose(coseno(a, [-4, 3]), 0.0) and np.isclose(coseno(a, -a), -1.0)   # coincide con el cálculo a mano

### ¿Qué observas?

`cos(a, b) = 0,96`: casi la misma dirección. `(−4, 3)` es perpendicular a `a` (producto escalar 0 → coseno 0) y `−a` apunta al revés (−1). **El coseno solo mira la dirección, no la longitud**: `(3, 4)` y `(30, 40)` tienen coseno 1. Por eso se usa con embeddings, donde la longitud del vector depende sobre todo de la frecuencia de la palabra.

---
## 2. Hipótesis distribucional: del vector disperso al denso

*"Conocerás una palabra por la compañía que tiene"* (Firth, 1957). Si dos palabras aparecen entre las mismas vecinas, tendrán un significado parecido. La versión más simple: para cada palabra, un vector con **un recuento por cada palabra del vocabulario** (cuántas veces aparece esa palabra justo al lado). Es un vector **disperso**: tantas dimensiones como palabras, casi todo ceros.

Mini-corpus de 4 frases, ventana de 1 palabra a cada lado. A mano: los contextos de `gato` son `el` (2 veces: en la frase 1 y en la 3), `bebe` (1) y `duerme` (1); los de `perro` son exactamente los mismos, luego `cos(gato, perro) = 1`. `leche` solo aparece junto a `bebe` (1 vez): `cos(gato, leche) = 1/√6 ≈ 0,408`.

In [ ]:
# datos-es
# Mini-corpus de 4 frases (ventana de 1 palabra a cada lado)
MINI = [["el", "gato", "bebe", "leche"], ["el", "perro", "bebe", "agua"], ["el", "gato", "duerme"], ["el", "perro", "duerme"]]
VOCAB = ["el", "gato", "perro", "bebe", "duerme", "leche", "agua"]

In [ ]:
indice = {p: i for i, p in enumerate(VOCAB)}
M = np.zeros((len(VOCAB), len(VOCAB)), dtype=int)        # fila = palabra, columna = palabra de contexto
for frase in MINI:
    for i, p in enumerate(frase):
        for j in (i - 1, i + 1):                          # ventana de 1 a cada lado
            if 0 <= j < len(frase):
                M[indice[p], indice[frase[j]]] += 1
print(pd.DataFrame(M, index=VOCAB, columns=VOCAB))
cos_gp = coseno(M[indice["gato"]], M[indice["perro"]])
cos_gl = coseno(M[indice["gato"]], M[indice["leche"]])
print("cos(gato, perro) =", round(cos_gp, 3), "| cos(gato, leche) =", round(cos_gl, 3))
assert M[indice["gato"], 0] == 2 and np.isclose(cos_gp, 1.0) and np.isclose(cos_gl, 1 / np.sqrt(6))   # coincide con el cálculo a mano

### ¿Qué observas?

La matriz tiene 7 × 7 = 49 casillas para un vocabulario de 7 palabras. Con un vocabulario real de unas 10 000 palabras serían **10 000 dimensiones por palabra**, casi todas cero. `gato` y `perro` salen idénticos porque en este mini-corpus juegan el mismo papel. Los **embeddings densos** (Word2Vec, fastText) comprimen esa información en unas decenas de dimensiones (aquí, 50) que se **aprenden** prediciendo palabras del contexto; a cambio, las dimensiones ya no significan nada por separado y el resultado depende del azar del entrenamiento.

---
## 3. Los datos: 3-4 obras por idioma y un corpus moderno diminuto

- **Español:** *Don Quijote* (1605-1615), *La Regenta* (1884-1885), *Marianela* (1878) y el *Lazarillo de Tormes* (1554).
- **Italiano:** *I promessi sposi* (1840-1842), *Le avventure di Pinocchio* (1883), *La vita militare* (ed. de 1869) y *Novelle* (ed. de 1884).
- Todas de dominio público (Project Gutenberg), unidas en un archivo por idioma. Cada línea se pasa a minúsculas, se corta en frases (`. ! ? ; :` y saltos de línea) y se queda solo con las **letras**: las cifras y la puntuación desaparecen (y con ellas los apóstrofos: `dell'uomo` da `dell` y `uomo`; una simplificación que declaramos).
- **Corpus moderno sintético:** el `texto_limpio` de las reseñas y los tickets del curso (700 textos por idioma). Es **diminuto** (unas 11 000 palabras): con eso un modelo de embeddings casi no aprende. Sirve solo para ver el contraste de registro.

In [ ]:
# datos-es
ARCHIVOS = {"es": "gutenberg/embeddings_es_obras.txt", "it": "gutenberg/embeddings_it_obras.txt"}
# Palabras de consulta fijadas ANTES de mirar los resultados (no se eligen los vecinos "bonitos")
CONSULTAS = {"es": ["rey", "amor", "dinero", "mujer", "muerte", "caballero", "casa", "dios"],
             "it": ["re", "amore", "denaro", "donna", "morte", "cavaliere", "casa", "dio"]}
GENERO = {"es": ("mujer", "hombre"), "it": ("donna", "uomo")}
ANALOGIAS = {"es": [("rey", "hombre", "mujer", "reina"), ("padre", "hombre", "mujer", "madre"), ("señor", "hombre", "mujer", "señora")],
             "it": [("re", "uomo", "donna", "regina"), ("padre", "uomo", "donna", "madre"), ("signore", "uomo", "donna", "signora")]}
POLISEMIA = {"es": ["vino", "cabo"], "it": ["porta", "sale"]}
FUERA_DE_VOCABULARIO = {"es": ["caballeroo", "cabalero", "quijotesco", "hidalguez"],
                        "it": ["cavalliere", "prommesso", "signorotti", "spaventosissimo"]}

In [ ]:
FRASES, MODERNO = {}, {}
for idioma, ruta in ARCHIVOS.items():
    with open(cargar_datos(ruta), encoding="utf-8") as f:
        FRASES[idioma] = frases_de_texto(f.read())
    MODERNO[idioma] = frases_modernas(idioma)
    print(idioma, "| literario: frases", len(FRASES[idioma]), "| palabras", sum(map(len, FRASES[idioma])),
          "| moderno: frases", len(MODERNO[idioma]), "| palabras", sum(map(len, MODERNO[idioma])))
    assert sum(map(len, FRASES[idioma])) > 400_000 and sum(map(len, MODERNO[idioma])) < 15_000

---
## 4. Word2Vec skip-gram: vecinos más próximos

**Skip-gram** entrena una red pequeña para **predecir las palabras del contexto a partir de la palabra central**; los vectores son los pesos aprendidos. Parámetros que fijamos (*ejemplo ilustrativo*): `vector_size=50`, `window=5` (hasta 5 palabras a cada lado), `min_count=5` (se ignoran las palabras que aparecen menos de 5 veces), `epochs=5`, `workers=1` y una `seed` fija (con un solo hilo el resultado es reproducible; con varios, no).

Las 8 palabras de consulta están **fijadas de antemano** en la celda de datos: no elegimos las que salen bien. Mira los resultados con espíritu crítico.

> **Nota:** al entrenar puede aparecer en rojo un mensaje como `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`. No es un error de tu código ni afecta al resultado: es un aviso cosmético de la extensión compilada en C de gensim (ligado a cómo libera memoria en algunas combinaciones de sistema operativo y librería BLAS). Se puede ignorar con seguridad.

In [ ]:
BASE = {}
for idioma in FRASES:
    BASE[idioma] = entrenar_w2v(FRASES[idioma], sg=1, window=5, seed=1)
    print("=" * 8, idioma, "| vocabulario (min_count=5):", len(BASE[idioma].wv), "| dimensión:", BASE[idioma].wv.vector_size)
    for palabra in CONSULTAS[idioma]:
        vec = BASE[idioma].wv.most_similar(palabra, topn=6)
        print(f"{palabra:>10} ->", ", ".join(f"{w} ({s:.2f})" for w, s in vec))
    assert len(BASE[idioma].wv) > 5000

### ¿Qué observas?

Hay resultados **razonables** (`caballero` → `andante`, `hidalgo`; `donna` → `signora`, `ragazza`; `casa` → `chiesa`, `camera`) y otros **malos o arcaicos**: `dios` → `vuesa`, `suplico` (formas de tratamiento del siglo XVII), `dinero` → `testamento`, `sabéis`, `dio` → `vossignoria`, `re` → `podestà`, `marchese` (cargos del Milán del siglo XVII). Los vecinos son **palabras que aparecen en contextos parecidos**, no sinónimos: `amor` sale junto a `desprecio` (opuesto, mismo contexto) y `muerte` junto a `pobreza`. Y la similitud **no mide calidad**: `denaro` → `ohi`, `dirti`, `melampo` con similitud 0,98 (basura), mientras que `caballero` → `andante` sale con 0,90. Las cifras dependen del corpus y de la semilla: **no son "probabilidades de acierto"**.

---
## 5. ¿Cuánto de esto es ruido? Otras semillas, CBOW y otra ventana

Entrenamos de nuevo cambiando **una sola cosa** cada vez y medimos cuántos de los 10 vecinos más próximos se conservan (`solape`: 1 = idénticos, 0 = ninguno), en las 8 palabras de consulta y en las 200 palabras más frecuentes (de al menos 4 letras).

- **Otra semilla** (`seed=2`): mismo algoritmo, mismos datos.
- **CBOW** (`sg=0`): predice la palabra central a partir del contexto; es más rápido y suele ir bien con palabras frecuentes.
- **Ventana 2** y **ventana 8**: una ventana corta capta relaciones más sintácticas (palabras intercambiables), una larga más temáticas (palabras del mismo asunto).

In [ ]:
VARIANTES = {"seed 2": dict(sg=1, window=5, seed=2), "CBOW": dict(sg=0, window=5, seed=1),
             "window 2": dict(sg=1, window=2, seed=1), "window 8": dict(sg=1, window=8, seed=1)}
OTROS = {idioma: {n: entrenar_w2v(FRASES[idioma], **p) for n, p in VARIANTES.items()} for idioma in FRASES}
tabla = []
for idioma in FRASES:
    frec = frecuentes(FRASES[idioma], BASE[idioma])
    for nombre, modelo in OTROS[idioma].items():
        tabla.append([idioma, nombre, round(solape(BASE[idioma], modelo, CONSULTAS[idioma]), 2), round(solape(BASE[idioma], modelo, frec), 2)])
res = pd.DataFrame(tabla, columns=["idioma", "variante", "solape (8 consultas)", "solape (200 frecuentes)"])
print(res.to_string(index=False))
res.pivot(index="variante", columns="idioma", values="solape (200 frecuentes)").plot.bar(rot=0)
plt.ylim(0, 1); plt.ylabel("fracción de los 10 vecinos que se conserva")
plt.title("Solape con el modelo base (skip-gram, ventana 5, semilla 1)")
plt.show()
assert (res.iloc[:, 3] < 1).all() and (res.iloc[:, 3] > 0).all()

### ¿Qué observas?

Solo cambiar la semilla hace que se pierda alrededor del **40 %** de los vecinos en las 200 palabras frecuentes (solape ≈ 0,58) y más aún en las 8 de consulta (0,50 en español y 0,38 en italiano; al ser pocas, la cifra es más irregular). Cambiar la ventana lo empeora (≈ 0,44-0,53) y pasar a CBOW aún más (≈ 0,25). **No hay un "vecino correcto"**: hay una lista que depende de decisiones y del azar. Con más texto (millones de palabras) el solape entre semillas sube; con unos cientos de miles de palabras, como aquí, es lo que ves. Regla práctica: **antes de interpretar un vecino, comprueba que aparece con otra semilla**.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma, "| primeros 6 vecinos de las 3 primeras consultas, semilla 1 frente a semilla 2")
    for palabra in CONSULTAS[idioma][:3]:
        print(f"{palabra:>10} s1:", vecinos(BASE[idioma], palabra), "| s2:", vecinos(OTROS[idioma]["seed 2"], palabra))

---
## 6. Analogías: una demostración de los límites

La analogía famosa es `rey − hombre + mujer ≈ reina`. Con corpus enormes a veces funciona; con un corpus pequeño y antiguo **no hay por qué esperar que funcione**. La usamos para medir en qué **posición** queda la respuesta esperada entre todos los candidatos (1 = la primera). Tres analogías fijadas de antemano por idioma.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma)
    for a, b, c_, objetivo in ANALOGIAS[idioma]:
        posicion, primero = rango_analogia(BASE[idioma], a, b, c_, objetivo)
        if posicion is None:
            print(f"{a} - {b} + {c_} -> '{objetivo}' no está en el vocabulario")
        else:
            print(f"{a} - {b} + {c_} -> esperada '{objetivo}': posición {posicion} de {len(BASE[idioma].wv) - 3} | primer candidato: '{primero}'")

### ¿Qué observas?

Los resultados son **desiguales**: `padre → madre` queda primera en español, `señor → señora` cuarta (la primera es `sobrina`), `rey → reina` en la posición 61 de 10 143 (la primera es `hacienda`) y en italiano `re → regina` ni siquiera puede probarse porque la palabra esperada no está en el vocabulario. Eso es lo esperable con este tamaño. Una analogía que sale bien en un ejemplo **no demuestra** que el modelo "entienda" nada: habría que probar muchas y contar los aciertos (con corpus mucho mayores se hace así, y el acierto está lejos del 100 %). Si ves analogías perfectas en una demo, sospecha de que se eligieron entre muchas.

---
## 7. fastText: palabras raras, fuera de vocabulario y erratas

**fastText** (Bojanowski et al., 2017) representa cada palabra como la **suma de los vectores de sus n-gramas de caracteres** (aquí, de 3 a 5, con marcas de inicio y fin: `<ca`, `cab`, `aba`…). Dos consecuencias: (1) una palabra **rara** comparte n-gramas con otras de la misma familia, luego su vector se apoya en ellas; (2) una palabra **que no está en el vocabulario** (una errata, una forma derivada) **sí recibe vector**, a partir de sus n-gramas. Word2Vec, en cambio, lanza un error (`KeyError`). Es muy útil en lenguas de morfología rica (español, italiano: muchas formas por verbo).

Elegimos las palabras raras **con una regla**, no a mano: las tres primeras, por orden alfabético, con 5 o 6 apariciones y al menos 8 letras.

In [ ]:
FT = {idioma: entrenar_ft(FRASES[idioma], sg=1, window=5, seed=1) for idioma in FRASES}
for idioma in FRASES:
    print("=" * 8, idioma, "| vocabulario fastText:", len(FT[idioma].wv))
    cuenta = Counter(w for f in FRASES[idioma] for w in f)
    raras = sorted(w for w, n in cuenta.items() if n in (5, 6) and len(w) >= 8)[:3]
    for w in raras:
        print(f"{w:>14} ({cuenta[w]} veces) word2vec:", vecinos(BASE[idioma], w, 5), "| fastText:", vecinos(FT[idioma], w, 5))
    print("-" * 8, "palabras fuera de vocabulario y erratas")
    for w in FUERA_DE_VOCABULARIO[idioma]:
        en_w2v = w in BASE[idioma].wv.key_to_index
        print(f"{w:>14} ¿en el vocabulario de word2vec? {en_w2v} | fastText:", vecinos(FT[idioma], w, 5))
    try:
        BASE[idioma].wv[FUERA_DE_VOCABULARIO[idioma][0]]
    except KeyError as error:
        print("word2vec ->", type(error).__name__)
    assert FT[idioma].wv[FUERA_DE_VOCABULARIO[idioma][0]].shape == (50,)

### ¿Qué observas?

En las palabras raras, **word2vec da vecinos casi aleatorios** (una palabra vista 5 veces casi no tiene contexto) y **fastText da vecinos morfológicos** (`abdicación` → `adulación`, `educación`, `abnegación`; `abbracciato` → `abbandonato`, `afferrato`): comparten terminación, no necesariamente significado. Con las erratas, fastText acierta cuando la errata conserva muchos n-gramas (`cabalero` → `cabal`, `caballero`; `cavalliere` → `cavaliere`) y falla cuando no (`quijotesco` → `quijote` sí, pero también `galeote`, `dote`). **Es forma, no sentido**: fastText no sabe que `hidalguez` significa algo; solo ve trozos parecidos.

---
## 8. Límites: un vector por palabra y sesgos del corpus

**Un vector por palabra.** `vino` es el verbo *venir* y también la bebida; `porta` (italiano) es la puerta y el verbo *portare*. El modelo tiene **un solo vector** que mezcla todos los usos, ponderados por su frecuencia en el corpus.

**Sesgos.** El modelo refleja lo que hay en los textos. Preguntamos por los vecinos de la dirección `mujer − hombre` (y al revés): salen las palabras cuyo contexto se parece más al de una que al de otra **en estas novelas**, no una verdad sobre el mundo.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma)
    for w in POLISEMIA[idioma]:
        print(f"{w:>8} ({sum(1 for f in FRASES[idioma] for x in f if x == w)} veces) ->", vecinos(BASE[idioma], w, 8))
    a, b = GENERO[idioma]
    print(f"{a} - {b}:", [w for w, _ in BASE[idioma].wv.most_similar(positive=[a], negative=[b], topn=8)])
    print(f"{b} - {a}:", [w for w, _ in BASE[idioma].wv.most_similar(positive=[b], negative=[a], topn=8)])

### ¿Qué observas?

Para `vino` y `porta` salen vecinos de **un solo sentido** (el verbo en `vino`: `llegando`, `partió`; la puerta en `porta`: `soglia`, `finestra`): el otro queda **borrado**. `sale` (21 apariciones) da una lista sin sentido claro: pocas apariciones, pocos contextos. En las direcciones de género aparecen sobre todo **parentescos, tratamientos y adjetivos de las novelas** (`madre`, `hijos`, `marquesa`, `señorita`, `sobrina`; `caballero`, `famoso`, `santo`) y algún residuo sin sentido (`ningún`, `ese`; en italiano `uomo − donna` da `avvenire`, `oggetto`, `esercito`: casi ningún patrón): es el retrato de una literatura del siglo XIX y anterior, no un juicio. Con embeddings entrenados con textos de hoy y de internet los sesgos son otros, y siguen ahí: **un embedding hereda los sesgos de su corpus**.

---
## 9. Corpus literario frente a corpus sintético moderno

Entrenamos skip-gram sobre las reseñas y los tickets del curso (unas 11 000 palabras; `min_count=3` y 30 épocas para que quede algo de vocabulario) y lo comparamos con el modelo literario. Medimos primero **cuánto del vocabulario moderno conoce el modelo literario** (cobertura de palabras, no de tipos) y, después, los vecinos de las 5 primeras palabras frecuentes del corpus moderno de al menos 6 letras que existen en ambos.

In [ ]:
for idioma in FRASES:
    mod = entrenar_w2v(MODERNO[idioma], sg=1, window=5, seed=1, epochs=30, min_count=3)
    cuenta = Counter(w for f in MODERNO[idioma] for w in f)
    total = sum(cuenta.values())
    cubiertas = sum(n for w, n in cuenta.items() if w in BASE[idioma].wv.key_to_index)
    ausentes = [w for w, n in cuenta.most_common(60) if w not in BASE[idioma].wv.key_to_index and len(w) >= 6][:6]
    comunes = [w for w, n in cuenta.most_common(300) if w in mod.wv.key_to_index and w in BASE[idioma].wv.key_to_index and len(w) >= 6][:5]
    print("=" * 8, idioma, "| vocabulario moderno:", len(mod.wv), "| cobertura del vocabulario literario sobre las palabras del texto moderno:", round(cubiertas / total, 2))
    print("Frecuentes en el corpus moderno y ausentes del literario:", ausentes)
    for w in comunes:
        print(f"{w:>12} literario:", vecinos(BASE[idioma], w, 5), "| moderno:", vecinos(mod, w, 5))
    assert 0.5 < cubiertas / total < 0.95 and len(comunes) == 5

### ¿Qué observas?

Las palabras del dominio moderno (`paquete`, `devolución`, `factura`, `tarjeta`; `fattura`, `addebito`, `rimborso`) **no existen** en el vocabulario literario: un modelo entrenado con Cervantes no puede representarlas. La cobertura global parece alta (0,81 en español y 0,77 en italiano) porque la aportan las palabras gramaticales; **el 19-23 % que falta es sobre todo vocabulario del dominio**. En las palabras comunes, los vecinos literarios son **verbos y arcaísmos** (`necesito` → `pregunto`, `confieso`), los del corpus moderno son de **trámites** (`necesito` → `quisiera`, `extracto`): el mismo texto de la palabra, dos usos. Pero el modelo moderno se entrena con ~11 000 palabras: sus vecinos son **muy poco fiables** (y con otra semilla cambiarían). Conclusión práctica: para un dominio concreto, o hay que entrenar con **mucho** texto de ese dominio, o partir de vectores **preentrenados** y comprobar que cubren el vocabulario.

---
## Resumen

- **Hipótesis distribucional:** el significado se aproxima por el contexto. Los embeddings son vectores **densos** aprendidos a partir de los contextos; el **coseno** compara direcciones.
- **Word2Vec** (skip-gram/CBOW) y **fastText** se entrenan con `gensim`; la **ventana**, `min_count`, `vector_size`, las épocas y la **semilla** cambian el resultado.
- Con **unos cientos de miles de palabras** los vecinos son **ruidosos**: comprueba siempre con otra semilla. Las **analogías** son una demostración de los límites, no un resultado.
- **fastText** da vector a palabras raras, fuera de vocabulario y erratas, pero por **forma**, no por sentido.
- Límites: **un vector por palabra**, **sesgos** heredados del corpus, cobertura del dominio.

**Tarea:** el notebook de práctica (`clase10_practica_word_embeddings.ipynb`). **Siguiente:** clase 11, búsqueda y recuperación clásica (TF-IDF, BM25): el recuperador que los embeddings densos complementan.